<div style="text-align: center; font-size: 40px; font-weight: bold;">
    Train Model Deeplabv3+ Architecture
</div>

# Libraries

In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
IMG_SIZE = (512, 512)

def load_image(img_path, mask_path):
    # ----- image -----
    img = tf.io.read_file(img_path)
    img = tf.image.decode_bmp(img, channels=3)   # gambar input RGB
    img = tf.image.resize(img, (512, 512))
    img = tf.cast(img, tf.float32) / 255.0

    # ----- mask -----
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_bmp(mask, channels=0)   # biarkan sesuai file
    mask = tf.image.resize(mask, (512, 512), method="nearest")
    
    # pastikan hanya 1 channel
    if tf.shape(mask)[-1] != 1:
        mask = tf.image.rgb_to_grayscale(mask)

    # ubah ke {0,1}
    mask = tf.cast(mask > 127, tf.float32)

    return img, mask


def make_dataset(img_dir, mask_dir, batch_size=4, shuffle=True):
    img_files = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir)])
    mask_files = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])
    ds = tf.data.Dataset.from_tensor_slices((img_files, mask_files))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=100)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset("Data_split/train/images", "Data_split/train/masks", batch_size=8)
val_ds   = make_dataset("Data_split/valid/images", "Data_split/valid/masks", batch_size=8, shuffle=False)



2025-10-06 21:35:00.258801: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2025-10-06 21:35:00.258825: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-10-06 21:35:00.258831: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-10-06 21:35:00.259201: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-10-06 21:35:00.259227: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
def sepconv_bn_relu(x, filters, kernel=3, dilation=1, name=None):
    """SeparableConv2D + BN + ReLU"""
    x = layers.SeparableConv2D(filters, kernel,
                               padding="same",
                               use_bias=False,
                               dilation_rate=dilation,
                               depthwise_initializer="he_normal",
                               pointwise_initializer="he_normal",
                               name=None if name is None else f"{name}_sepconv")(x)
    x = layers.BatchNormalization(name=None if name is None else f"{name}_bn")(x)
    x = layers.ReLU(name=None if name is None else f"{name}_relu")(x)
    return x

def aspp(x, out_channels=256, rates=(6, 12, 18), name="aspp"):
    # branch 1: 1x1
    b0 = layers.Conv2D(out_channels, 1, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    b0 = layers.BatchNormalization()(b0); b0 = layers.ReLU()(b0)

    # branch 2..4: atrous separable conv
    b1 = sepconv_bn_relu(x, out_channels, 3, dilation=rates[0])
    b2 = sepconv_bn_relu(x, out_channels, 3, dilation=rates[1])
    b3 = sepconv_bn_relu(x, out_channels, 3, dilation=rates[2])

    # branch 5: image pooling
    gp = layers.GlobalAveragePooling2D()(x)
    gp = layers.Reshape((1, 1, x.shape[-1]))(gp)
    gp = layers.Conv2D(out_channels, 1, padding="same", use_bias=False)(gp)
    gp = layers.BatchNormalization()(gp); gp = layers.ReLU()(gp)
    gp = tf.image.resize(gp, (tf.shape(x)[1], tf.shape(x)[2]), method="bilinear")

    # concat semua branch
    y = layers.Concatenate()([b0, b1, b2, b3, gp])
    y = layers.Conv2D(out_channels, 1, padding="same", use_bias=False, kernel_initializer="he_normal")(y)
    y = layers.BatchNormalization()(y); y = layers.ReLU()(y)
    y = layers.Dropout(0.1)(y)
    return y

# ---------- DeepLabv3+ Original ----------
def DeepLabV3Plus_Xception(input_shape=(512, 512, 3), num_classes=1):
    """
    DeepLabv3+ (original-style):
    - Backbone: Xception pretrained (ImageNet)
    - Output stride ~16
    - ASPP: (6,12,18)
    - Decoder: low-level (48 ch) + 2× sepconv 256
    """
    inputs = layers.Input(shape=input_shape)

    # backbone
    base = tf.keras.applications.Xception(input_tensor=inputs,
                                          include_top=False,
                                          weights="imagenet")

    # low-level feature (1/4 resolusi)
    low = base.get_layer("block2_sepconv2_act").output

    # high-level feature (1/16 resolusi)
    high = base.get_layer("block13_sepconv2_bn").output

     # ASPP
    x = aspp(high, out_channels=256, rates=(6, 12, 18), name="aspp")

    # upsample ke ukuran low-level (gunakan resize agar match)
    x = tf.image.resize(x, size=(tf.shape(low)[1], tf.shape(low)[2]), method="bilinear")

    # low-level projection
    low_proj = layers.Conv2D(48, 1, padding="same", use_bias=False)(low)
    low_proj = layers.BatchNormalization()(low_proj); low_proj = layers.ReLU()(low_proj)

    # concat
    x = layers.Concatenate()([x, low_proj])

    # decoder refinement
    x = sepconv_bn_relu(x, 256, 3, name="decoder1")
    x = sepconv_bn_relu(x, 256, 3, name="decoder2")

    # resize ke ukuran input (pastikan sama persis)
    x = tf.image.resize(x, size=(input_shape[0], input_shape[1]), method="bilinear")

    # output layer
    logits = layers.Conv2D(num_classes, 1, padding="same")(x)
    act = "sigmoid" if num_classes == 1 else "softmax"
    outputs = layers.Activation(act)(logits)

    return models.Model(inputs, outputs, name="DeepLabV3plus_Xception")

In [4]:
model = DeepLabV3Plus_Xception(input_shape=(512, 512, 3), num_classes=1)
model.summary()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
               loss="binary_crossentropy",
               metrics=["accuracy"])

Model: "DeepLabV3plus_Xception"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 512, 512, 3)]        0         []                            
                                                                                                  
 block1_conv1 (Conv2D)       (None, 255, 255, 32)         864       ['input_1[0][0]']             
                                                                                                  
 block1_conv1_bn (BatchNorm  (None, 255, 255, 32)         128       ['block1_conv1[0][0]']        
 alization)                                                                                       
                                                                                                  
 block1_conv1_act (Activati  (None, 255, 255, 32)         0         ['block1_

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=100, verbose=2)

Epoch 1/100


2025-10-06 21:35:03.560747: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
